# Wildfire classification and Grad-CAM

This notebook runs the same packaged code used by DVC, MLflow, the CLI, and the API. It preserves the supplied train/validation/test split, classifies test images with the validation-selected threshold, and visualizes which final DenseNet feature regions influenced each decision.

> Run from the project environment created in `README.md`. Training the full 30,250-image training set is best done on a GPU.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'params.yaml').exists():
    raise FileNotFoundError('Start Jupyter in wildfire_project/ or wildfire_project/notebooks/.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_ml.config import load_config
from wildfire_ml.data import prepare_data
from wildfire_ml.gradcam import load_image_batch, make_gradcam_heatmap, overlay_heatmap

config = load_config(PROJECT_ROOT / 'params.yaml')
class_names = config.section('data')['class_names']
image_size = tuple(config.section('data')['image_size'])
class_names, image_size

## 1. Prepare and inspect the supplied data

The operation is safe and idempotent. On the first run it extracts about 42,850 images and verifies that every image is readable.

In [ ]:
summary = prepare_data(config)
summary

In [ ]:
processed_dir = config.path('data', 'processed_dir')
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, class_name in enumerate(class_names):
    examples = sorted((processed_dir / 'train' / class_name).glob('*.jpg'))[:4]
    for axis, path in zip(axes[row], examples):
        axis.imshow(Image.open(path).convert('RGB'))
        axis.set_title(class_name)
        axis.axis('off')
plt.tight_layout()

## 2. Train and evaluate (optional)

Leave `RUN_TRAINING=False` when `artifacts/models/final.keras` already exists. The training call performs the frozen-head phase and controlled fine-tuning; evaluation selects F2 threshold on validation and reports the untouched test split.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from wildfire_ml.training import train
    from wildfire_ml.evaluation import evaluate

    training_summary = train(config)
    evaluation_summary = evaluate(config)
    print(evaluation_summary['test'])

### 2a. Recover from `best.weights.h5` after an interrupted run

Use this cell instead of rerunning training. Upload or copy `best.weights.h5`, then set `CHECKPOINT_PATH` to its location. The checkpoint must have been produced by this notebook's DenseNet architecture and `params.yaml`. Loading is strict, so an incompatible checkpoint fails rather than silently loading only part of it.

This reconstructs the model without downloading ImageNet weights, loads every saved model weight, and writes the complete `artifacts/models/final.keras` file expected by the remaining cells. A weights-only checkpoint is safe for evaluation and prediction, but it cannot restore the previous optimizer state or exact training epoch.

In [ ]:
RECOVER_FROM_CHECKPOINT = True
CHECKPOINT_PATH = config.path('paths', 'model_dir') / 'best.weights.h5'
# If the notebook uploader placed it elsewhere, use the actual path, for example:
# CHECKPOINT_PATH = Path('/content/best.weights.h5')  # Google Colab
# CHECKPOINT_PATH = PROJECT_ROOT / 'best.weights.h5'  # Jupyter project root

if RECOVER_FROM_CHECKPOINT:
    import tensorflow as tf
    from wildfire_ml.model import build_model

    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')
    tf.keras.backend.clear_session()
    recovery_model_config = dict(config.section('model'))
    recovery_model_config['imagenet_weights'] = None  # all values are replaced below
    recovered_model = build_model(recovery_model_config, image_size=image_size)
    recovered_model.load_weights(str(CHECKPOINT_PATH))  # strict architecture check

    final_model_path = config.path('paths', 'model_dir') / 'final.keras'
    final_model_path.parent.mkdir(parents=True, exist_ok=True)
    recovered_model.save(final_model_path)
    print(f'Recovered deployable model: {final_model_path}')

### 2b. Evaluate the recovered checkpoint

This performs inference only—no additional training epochs. It selects the F2 operating threshold from validation predictions, freezes that threshold, and then evaluates the untouched test split. Processing 12,600 images can still take several minutes on CPU.

In [ ]:
RUN_EVALUATION = True

if RUN_EVALUATION:
    from wildfire_ml.evaluation import evaluate

    evaluation_summary = evaluate(config)
    print('Validation-selected threshold:', evaluation_summary['threshold_selected_on_validation'])
    display(evaluation_summary['test'])

## 3. Load the deployable model and its approved threshold

In [ ]:
import tensorflow as tf

model_path = config.path('paths', 'model_dir') / 'final.keras'
if not model_path.exists():
    raise FileNotFoundError(
        f'{model_path} is missing. Set RUN_TRAINING=True above or run: dvc repro'
    )
model = tf.keras.models.load_model(model_path, compile=False)

metrics_path = config.path('paths', 'reports_dir') / 'metrics.json'
if metrics_path.exists():
    evaluation_summary = json.loads(metrics_path.read_text(encoding='utf-8'))
    threshold = float(evaluation_summary['threshold_selected_on_validation'])
    print('Validation-selected threshold:', round(threshold, 4))
    print('Test metrics:')
    display(evaluation_summary['test'])
else:
    threshold = 0.5
    print('Evaluation report not found; using threshold=0.5 for demonstration.')

## 4. Classify unseen test images

The model accepts raw RGB pixel values; DenseNet preprocessing is embedded in the saved model.

In [ ]:
test_paths = (
    sorted((processed_dir / 'test' / 'nowildfire').glob('*.jpg'))[:2]
    + sorted((processed_dir / 'test' / 'wildfire').glob('*.jpg'))[:2]
)

fig, axes = plt.subplots(1, len(test_paths), figsize=(16, 4))
predictions = []
for axis, path in zip(axes, test_paths):
    batch, original = load_image_batch(path, image_size)
    probability = float(model(batch, training=False).numpy()[0, 0])
    predicted = 'wildfire' if probability >= threshold else 'nowildfire'
    truth = path.parent.name
    predictions.append({
        'file': path.name, 'truth': truth, 'prediction': predicted,
        'wildfire_probability': probability
    })
    axis.imshow(original)
    axis.set_title(f'true: {truth}\npred: {predicted} ({probability:.1%})')
    axis.axis('off')
plt.tight_layout()
predictions

## 5. Explain one decision with Grad-CAM

The explained class follows the validation-selected operational threshold. Pass `target_class=1` to ask specifically what increased the wildfire score. Warm regions contribute most strongly.

In [ ]:
selected_path = test_paths[-1]
batch, original = load_image_batch(selected_path, image_size)
heatmap, probability, explained_class = make_gradcam_heatmap(
    batch, model, decision_threshold=threshold
)
overlay = overlay_heatmap(original, heatmap, alpha=0.4)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original)
axes[0].set_title(f'Input: {selected_path.parent.name}')
axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title('Grad-CAM feature map')
axes[2].imshow(overlay)
axes[2].set_title(
    f'Explains {class_names[explained_class]}\nP(wildfire)={probability:.1%}'
)
for axis in axes:
    axis.axis('off')
plt.tight_layout()

## Interpretation checklist

- Compare correct and incorrect predictions from both classes.
- Look for consistent attention to visible fire/smoke cues rather than borders, text, water, clouds, or collection artifacts.
- Treat Grad-CAM as a sensitivity visualization, not a fire segmentation mask.
- Review PR-AUC, recall, F2, confusion matrix, and calibration together; accuracy alone is not an operational safety measure.
- Revalidate by geography, season, sensor, and time before deployment.